In [7]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))


In [8]:

from src.data.ingest import build_universe, download_prices, clean_prices

tickers = build_universe()
cache = Path("../data/raw/prices.parquet")
raw_df = download_prices(tickers, cache)
clean_df, report = clean_prices(raw_df)


Loading cached data from ../data/raw/prices.parquet
Dropping 7 tickers: ['AAF.L', 'CCEP.L', 'EDV.L', 'HLN.L', 'MNG.L', 'MTLN.L', 'PSH.L']


/var/folders/c1/b4w8jwwx5gscsnykncp6_dym0000gn/T/ipykernel_57541/692558092.py:6: UserWarning: clean_prices: 2 ticker(s) have leading NaN after ffill (post-listing tickers): CTEC.L(467), AUTO.L(54). These cells are NOT bfilled to avoid look-ahead bias; downstream feature engineering will propagate NaN through rolling windows and drop the affected rows.
  clean_df, report = clean_prices(raw_df)


In [9]:
from src.features.engineer import build_features

features_df = build_features(clean_df)
print(features_df.shape)
print(features_df.head())
print(features_df.isnull().sum())


(271932, 9)
                   momentum  volatility_21d  drawdown_52w  relative_strength  \
date       ticker                                                              
2015-01-02 AAL.L        NaN             NaN           NaN                NaN   
           ABF.L        NaN             NaN           NaN                NaN   
           ADM.L        NaN             NaN           NaN                NaN   
           ALW.L        NaN             NaN           NaN                NaN   
           ANTO.L       NaN             NaN           NaN                NaN   

                   volume_ratio_20  beta_252        vix  forward_return  \
date       ticker                                                         
2015-01-02 AAL.L               NaN       NaN  17.790001       -0.051477   
           ABF.L               NaN       NaN  17.790001       -0.025047   
           ADM.L               NaN       NaN  17.790001        0.109506   
           ALW.L               NaN       NaN  17.790

In [10]:
features_clean = features_df.dropna()
print(features_clean.shape)
print(features_clean.index.get_level_values("date").min())


(246022, 9)
2015-12-24 00:00:00


In [11]:
features_clean.to_parquet("../data/processed/features.parquet")
print("Saved.")


Saved.


In [12]:
features_clean.columns

Index(['momentum', 'volatility_21d', 'drawdown_52w', 'relative_strength',
       'volume_ratio_20', 'beta_252', 'vix', 'forward_return',
       'forward_volatility'],
      dtype='object')

In [13]:
features_clean = features_df.dropna()
print(features_clean.shape)
print(features_clean.columns.tolist())
features_clean.to_parquet("../data/processed/features.parquet")


(246022, 9)
['momentum', 'volatility_21d', 'drawdown_52w', 'relative_strength', 'volume_ratio_20', 'beta_252', 'vix', 'forward_return', 'forward_volatility']


## Pre-commitment correlation audit (historical record)

The original feature set comprised 11 candidates. A pre-committed Spearman
correlation audit (median |ρ| across all tickers) flagged the following pairs
above the 0.7 threshold:

| Pair                                  | Median |ρ| | Decision                        |
|---------------------------------------|----------|---------------------------------|
| return_1m × rsi_14                    | 0.861    | drop return_1m                  |
| return_1m × relative_strength         | 0.856    | drop return_1m (consistent)     |
| rsi_14 × relative_strength            | 0.744    | drop rsi_14 (retained relative_strength for retail explainability) |
| drawdown_52w × ma_ratio_200           | 0.874    | drop ma_ratio_200               |
| drawdown_52w × percentile_52w         | 0.947    | drop percentile_52w             |
| ma_ratio_200 × percentile_52w         | 0.921    | drop ma_ratio_200 / percentile_52w (consistent) |

Borderline pair retained:
- ma_ratio_200 × rsi_14: 0.581 (below threshold; both already dropped above)

The final 7-feature set in `src/features/engineer.py` is the post-audit
result: momentum, volatility_21d, drawdown_52w, relative_strength,
volume_ratio_20, beta_252, vix. The audit is not re-runnable in this
notebook because the dropped candidates are no longer computed by
build_features(), the decisions are locked and this markdown cell is the
audit trail.